In [ ]:
import os, sys, torch, platform
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
for pkg in ["transformers", "peft", "bitsandbytes", "accelerate", "datasets", "huggingface_hub"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'installed')}")
    except Exception as e:
        print(f"{pkg}: ERROR {e}")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130 CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM GB: 39.49
transformers: 5.22.1
peft: 0.21.2
bitsandbytes: ERROR No module named 'bitsandbytes'
accelerate: 1.15.0
datasets: 4.8.5
huggingface_hub: 1.33.0
HF_TOKEN set: False
Drive mounted: False


In [ ]:
!pip install -q bitsandbytes
import os, bitsandbytes
print("bitsandbytes:", bitsandbytes.__version__)
assert os.environ.get("HF_TOKEN"), "HF_TOKEN must be set in environment or Colab Secrets"
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(token=os.environ["HF_TOKEN"])
info = api.model_info("google/gemma-2-2b-it")
print("Gemma-2-2b-it SHA:", info.sha)
assert info.sha == "299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8"
cfg = hf_hub_download("google/gemma-2-2b-it", "config.json", revision=info.sha, token=os.environ["HF_TOKEN"])
print("Verified config.json:", cfg)


bitsandbytes: 0.50.2
Gemma-2-2b-it SHA: 299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

Verified config.json: /root/.cache/huggingface/hub/models--google--gemma-2-2b-it/snapshots/299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8/config.json


In [ ]:
import hashlib, base64, lzma, json, pathlib, shutil
from collections import Counter

b64_data = open("/content/lz2.b64").read()
assert len(b64_data) == 39288, f"Expected 39288, got {len(b64_data)}"
bundle = json.loads(lzma.decompress(base64.b64decode(b64_data)).decode("utf-8"))

repo_root = pathlib.Path("/content/motherlypulse_fresh_repo")
if repo_root.exists():
    shutil.rmtree(repo_root)
repo_root.mkdir(parents=True, exist_ok=True)

for rel_path, content in bundle.items():
    norm_path = rel_path.replace("\\", "/")
    p = repo_root / norm_path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8", newline="\n")

expected_hashes = {
    "data/test_en.jsonl": "276f076f19d84600f840b05bb5e7cf583b7815ee11b39981ac6612950741b739",
    "data/test_pcm.jsonl": "fffdea0494d3f384264bf26527cf0c39d5ac97f8385b48ae61337d2a3720829f",
    "data/test_yo.jsonl": "c69cf9d77dc473148e6e6c4222aa0a4551c374bd93b2756d3bd0ed3f6ab64764",
    "MotherlyPulse_Data/04_Conversation_Data/fixed_15_evaluation_prompts.json": "7830ab3d7b8b0f601a464af34387675a53a279d59688e26592321f5b9529e7bc",
}
for rel_path, exp in expected_hashes.items():
    actual = hashlib.sha256((repo_root / rel_path).read_bytes()).hexdigest()
    assert actual == exp, f"Hash mismatch for {rel_path}: {actual} != {exp}"
    print(f"SHA256 OK: {rel_path} = {actual[:12]}...")

def norm_lang(val):
    v = (val or "").strip().lower()
    if v in ("en", "english"): return "en"
    if v in ("yo", "yoruba", "yorùbá"): return "yo"
    if v in ("pcm", "pidgin", "nigerian pidgin"): return "pcm"
    return v or "unknown"

eval_items = []
for lang, fname in [("en", "data/test_en.jsonl"), ("yo", "data/test_yo.jsonl"), ("pcm", "data/test_pcm.jsonl")]:
    for idx, line in enumerate((repo_root / fname).read_text(encoding="utf-8").splitlines()):
        if not line.strip():
            continue
        obj = json.loads(line)
        eval_items.append({
            "id": obj.get("pair_id", f"test_{lang}_{idx+1}"),
            "source": fname,
            "lang": lang,
            "triage": obj["triage_level"],
            "prompt": obj["prompt"],
            "expected": obj.get("target", ""),
        })

fixed_15 = json.loads((repo_root / "MotherlyPulse_Data/04_Conversation_Data/fixed_15_evaluation_prompts.json").read_text(encoding="utf-8"))
for item in fixed_15:
    eval_items.append({
        "id": item["id"],
        "source": "fixed_15_evaluation_prompts.json",
        "lang": norm_lang(item.get("lang")),
        "triage": item.get("expected_triage", item.get("triage", "unknown")),
        "prompt": item["prompt"],
        "expected": "",
    })

print(f"Total eval_items: {len(eval_items)}")
print("By lang:", dict(Counter(x["lang"] for x in eval_items)))
print("By triage:", dict(Counter(x["triage"] for x in eval_items)))
print("By source:", dict(Counter(x["source"] for x in eval_items)))


SHA256 OK: data/test_en.jsonl = 276f076f19d8...
SHA256 OK: data/test_pcm.jsonl = fffdea0494d3...
SHA256 OK: data/test_yo.jsonl = c69cf9d77dc4...
SHA256 OK: MotherlyPulse_Data/04_Conversation_Data/fixed_15_evaluation_prompts.json = 7830ab3d7b8b...
Total eval_items: 135
By lang: {'en': 45, 'yo': 45, 'pcm': 45}
By triage: {'Emergency': 42, 'Routine': 66, 'Priority': 27}
By source: {'data/test_en.jsonl': 40, 'data/test_yo.jsonl': 40, 'data/test_pcm.jsonl': 40, 'fixed_15_evaluation_prompts.json': 15}


In [ ]:
import os, sys, time, csv, pathlib, torch, gc, importlib.util
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

spec = importlib.util.spec_from_file_location("ai_connector", "/content/motherlypulse_fresh_repo/motherlypulse_app/ai_connector.py")
ai_connector = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ai_connector)
ClinicalInvariantValidator = ai_connector.ClinicalInvariantValidator

MODEL_ID = "google/gemma-2-2b-it"
MODEL_REV = "299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV, token=os.environ["HF_TOKEN"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REV,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"],
)
base_model.eval()
assert not hasattr(base_model, "peft_config"), "Expected pure base model without PEFT adapter!"
print(f"Loaded {MODEL_ID}@{MODEL_REV[:12]} | VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

validator = ClinicalInvariantValidator()

def run_eval(model, tok, items, out_csv_path, label="baseline"):
    pathlib.Path(out_csv_path).parent.mkdir(parents=True, exist_ok=True)
    rows = []
    t0_all = time.perf_counter()
    for i, item in enumerate(items):
        messages = [{"role": "user", "content": item["prompt"]}]
        formatted = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(formatted, return_tensors="pt").to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
            )
        latency = time.perf_counter() - t0
        gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
        response = tok.decode(gen_ids, skip_special_tokens=True).strip()
        val = validator.validate(response, item["triage"])
        clinical_violations = [v for v in val["violations"] if v != "missing_category_prefix"]
        contra_violations = [v for v in clinical_violations if v not in ("missing_emergency_escalation", "empty_response")]
        rows.append({
            "id": item["id"],
            "source": item["source"],
            "lang": item["lang"],
            "triage": item["triage"],
            "has_prefix": val["has_prefix"],
            "clinical_safe_excl_prefix": len(clinical_violations) == 0,
            "strict_safe": val["is_safe"],
            "escalates_112": val["escalates_112"],
            "has_contra_violation": len(contra_violations) > 0,
            "violations": "|".join(val["violations"]),
            "clinical_violations": "|".join(clinical_violations),
            "latency_s": round(latency, 4),
            "gen_tokens": int(len(gen_ids)),
            "char_len": len(response),
            "response": response,
        })
        if (i + 1) % 25 == 0 or (i + 1) == len(items):
            print(f"[{label}] {i+1}/{len(items)} done ({time.perf_counter()-t0_all:.1f}s)")
    df = pd.DataFrame(rows)
    df.to_csv(out_csv_path, index=False)
    return df

baseline_csv = "/content/fresh_chewie_1_2/baseline/chewie_1_2_fresh_baseline_eval.csv"
df_base = run_eval(base_model, tokenizer, eval_items, baseline_csv, label="baseline")
print(f"Saved {baseline_csv} ({len(df_base)} rows)")


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Loaded google/gemma-2-2b-it@299a8560bedf | VRAM allocated: 2.23 GB
[baseline] 25/135 done (157.5s)
[baseline] 50/135 done (314.4s)
[baseline] 75/135 done (471.7s)
[baseline] 100/135 done (624.6s)
[baseline] 125/135 done (781.4s)
[baseline] 135/135 done (839.5s)
Saved /content/fresh_chewie_1_2/baseline/chewie_1_2_fresh_baseline_eval.csv (135 rows)


In [ ]:
import hashlib, lzma, base64, json
import pandas as pd

df_base = pd.read_csv("/content/fresh_chewie_1_2/baseline/chewie_1_2_fresh_baseline_eval.csv")

def summarize_eval(df, name="eval"):
    n = len(df)
    em = df[df["triage"] == "Emergency"]
    summary = {
        "name": name,
        "n": int(n),
        "has_prefix_rate": round(float(df["has_prefix"].mean()), 4),
        "clinical_safe_excl_prefix_rate": round(float(df["clinical_safe_excl_prefix"].mean()), 4),
        "strict_safe_rate": round(float(df["strict_safe"].mean()), 4),
        "contra_violation_rate": round(float(df["has_contra_violation"].mean()), 4),
        "emergency_n": int(len(em)),
        "emergency_112_rate": round(float(em["escalates_112"].mean()), 4) if len(em) else None,
        "mean_latency_s": round(float(df["latency_s"].mean()), 4),
        "mean_gen_tokens": round(float(df["gen_tokens"].mean()), 2),
        "mean_char_len": round(float(df["char_len"].mean()), 2),
        "by_lang": {},
        "by_triage": {},
    }
    for lang, g in df.groupby("lang"):
        summary["by_lang"][lang] = {
            "n": int(len(g)),
            "has_prefix_rate": round(float(g["has_prefix"].mean()), 4),
            "clinical_safe_excl_prefix_rate": round(float(g["clinical_safe_excl_prefix"].mean()), 4),
            "strict_safe_rate": round(float(g["strict_safe"].mean()), 4),
            "contra_violation_rate": round(float(g["has_contra_violation"].mean()), 4),
        }
    for tr, g in df.groupby("triage"):
        summary["by_triage"][tr] = {
            "n": int(len(g)),
            "has_prefix_rate": round(float(g["has_prefix"].mean()), 4),
            "clinical_safe_excl_prefix_rate": round(float(g["clinical_safe_excl_prefix"].mean()), 4),
            "strict_safe_rate": round(float(g["strict_safe"].mean()), 4),
            "contra_violation_rate": round(float(g["has_contra_violation"].mean()), 4),
        }
    return summary

base_summary = summarize_eval(df_base, "baseline")
print("=== BASELINE SUMMARY ===")
print(json.dumps(base_summary, indent=2))

csv_bytes = open("/content/fresh_chewie_1_2/baseline/chewie_1_2_fresh_baseline_eval.csv", "rb").read()
print("BASELINE_CSV_SHA256:", hashlib.sha256(csv_bytes).hexdigest())
b64_lz = base64.b64encode(lzma.compress(csv_bytes)).decode("ascii")
print("BASELINE_CSV_LZMA_B64_START")
for i in range(0, len(b64_lz), 200):
    print(b64_lz[i:i+200])
print("BASELINE_CSV_LZMA_B64_END")


=== BASELINE SUMMARY ===
{
  "name": "baseline",
  "n": 135,
  "has_prefix_rate": 0.0,
  "clinical_safe_excl_prefix_rate": 0.8815,
  "strict_safe_rate": 0.0,
  "contra_violation_rate": 0.0148,
  "emergency_n": 42,
  "emergency_112_rate": 0.6667,
  "mean_latency_s": 6.2165,
  "mean_gen_tokens": 98.75,
  "mean_char_len": 346.46,
  "by_lang": {
    "en": {
      "n": 45,
      "has_prefix_rate": 0.0,
      "clinical_safe_excl_prefix_rate": 0.9778,
      "strict_safe_rate": 0.0,
      "contra_violation_rate": 0.0222
    },
    "pcm": {
      "n": 45,
      "has_prefix_rate": 0.0,
      "clinical_safe_excl_prefix_rate": 0.9111,
      "strict_safe_rate": 0.0,
      "contra_violation_rate": 0.0222
    },
    "yo": {
      "n": 45,
      "has_prefix_rate": 0.0,
      "clinical_safe_excl_prefix_rate": 0.7556,
      "strict_safe_rate": 0.0,
      "contra_violation_rate": 0.0
    }
  },
  "by_triage": {
    "Emergency": {
      "n": 42,
      "has_prefix_rate": 0.0,
      "clinical_safe_excl_pref

In [ ]:
import base64, gc, hashlib, json, lzma, os, torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Phase 4: Unload training model & reload fresh base + saved adapter
for _v in ["trainer", "peft_model", "train_model"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()
print(f"GPU allocated after training cleanup: {torch.cuda.memory_allocated()/1e9:.3f} GB")

ADAPTER_DIR = "/content/fresh_chewie_1_2/chewie_adapter"
POST_DIR = "/content/fresh_chewie_1_2/post_eval"
os.makedirs(POST_DIR, exist_ok=True)

for fn, expected in [
    ("adapter_model.safetensors", "2ae02447df703670dcb3fd1a0fc8d7fd35eac6ff106a3e1a9ef0de41e1af9efe"),
    ("adapter_config.json", "6f1fc1a5a275a25a6ddd71febf5bf7661b3899838f151bf9a4f29bb00f4c1427"),
]:
    actual = hashlib.sha256(open(os.path.join(ADAPTER_DIR, fn), "rb").read()).hexdigest()
    assert actual == expected, f"Mismatch for {fn}: {actual} != {expected}"
    print(f"Verified {fn} sha256={actual}")

print("adapter_config.json:")
print(open(os.path.join(ADAPTER_DIR, "adapter_config.json"), "r", encoding="utf-8").read())

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
fresh_base = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    revision="299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8",
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ.get("HF_TOKEN"),
)
ft_model = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)
ft_model.eval()
assert hasattr(ft_model, "peft_config"), "Expected peft_config on fine-tuned model"
print(f"Loaded PeftModel on {ft_model.device}, VRAM allocated={torch.cuda.memory_allocated()/1e9:.3f} GB")

# Phase 5: Post-training evaluation on identical 135 eval_items
ft_csv = f"{POST_DIR}/chewie_1_2_fresh_finetuned_eval.csv"
ft_df = run_eval(ft_model, tokenizer, eval_items, ft_csv, "finetuned")
ft_summary = summarize_eval(ft_df, "finetuned")
print("FINETUNED_SUMMARY:", json.dumps(ft_summary, indent=2))

csv_bytes = open(ft_csv, "rb").read()
print(f"Saved {ft_csv}: {len(csv_bytes)} bytes, sha256={hashlib.sha256(csv_bytes).hexdigest()}")
csv_b64 = base64.b64encode(lzma.compress(csv_bytes)).decode("ascii")
print("FINETUNED_CSV_LZMA_B64_START")
print(csv_b64)
print("FINETUNED_CSV_LZMA_B64_END")

GPU allocated after training cleanup: 0.017 GB
Verified adapter_model.safetensors sha256=2ae02447df703670dcb3fd1a0fc8d7fd35eac6ff106a3e1a9ef0de41e1af9efe
Verified adapter_config.json sha256=6f1fc1a5a275a25a6ddd71febf5bf7661b3899838f151bf9a4f29bb00f4c1427
adapter_config.json:
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "google/gemma-2-2b-it",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "kasa_config": null,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded PeftModel on cuda:0, VRAM allocated=2.270 GB
[finetuned] 25/135 done (104.1s)
[finetuned] 50/135 done (272.8s)
[finetuned] 75/135 done (503.0s)
[finetuned] 100/135 done (633.9s)
[finetuned] 125/135 done (755.1s)
[finetuned] 135/135 done (822.0s)
FINETUNED_SUMMARY: {
  "name": "finetuned",
  "n": 135,
  "has_prefix_rate": 0.8889,
  "clinical_safe_excl_prefix_rate": 0.8889,
  "strict_safe_rate": 0.7926,
  "contra_violation_rate": 0.0,
  "emergency_n": 42,
  "emergency_112_rate": 0.6429,
  "mean_latency_s": 6.0875,
  "mean_gen_tokens": 65.97,
  "mean_char_len": 209.56,
  "by_lang": {
    "en": {
      "n": 45,
      "has_prefix_rate": 0.8889,
      "clinical_safe_excl_prefix_rate": 0.9778,
      "strict_safe_rate": 0.8667,
      "contra_violation_rate": 0.0
    },
    "pcm": {
      "n": 45,
      "has_prefix_rate": 0.8889,
      "clinical_safe_excl_prefix_rate": 1.0,
      "strict_safe_rate": 0.8889,
      "contra_violation_rate": 0.0
    },
    "yo": {
      "n": 45,
      "has_p